### MFCCs extraction

In [ ]:
# read xlsx file
import pandas as pd
df = pd.read_excel('***********_clinicalinterpretation.xlsx')
df

In [ ]:
df['metadata'].value_counts()

In [ ]:
df['event_clinical_yesno'].value_counts()

In [ ]:
meta_full_speech = df[
    df['metadata'].str.contains('read_text|answer_question', na=False)
]

meta_full_speech

In [ ]:
len(meta_full_speech['sid'].unique())

In [ ]:
len(df['sid'].unique())

In [ ]:
import os
import pandas as pd
import librosa
from spafe.features.mfcc import mfcc
import numpy as np
from scipy.io.wavfile import read
from spafe.utils.preprocessing import SlidingWindow

def mfcc_features_extractor(sig, fs):
    #fs, sig = read(file)
    mfccs  = mfcc(sig,
              fs=fs,
              pre_emph=1,
              pre_emph_coeff=0.97,
              window=SlidingWindow(0.025, 0.005, "hamming"),
              nfilts=80,
              nfft=2048,
              low_freq=0,
              high_freq=fs/2,
              num_ceps=80,
             )
    mfccs_scaled_features = np.mean(mfccs,axis=0)
    
    return mfccs_scaled_features

root_dir = "*********/tacticas_audio/tacticas" 


result_df = pd.DataFrame()

#iterate though ROWS IN meta_full_speech
for index, row in meta_full_speech.iterrows():
    name = row['filename'].split('/')[-1]
    sid = row['sid']
    path = f'uid0000{sid}/{name}' 
    audio_path = os.path.join(root_dir, path)
    if os.path.exists(audio_path):
        # Read audio file
        audio, sr = librosa.load(audio_path)
        
        # Extract features
        features = mfcc_features_extractor(audio, sr)
        
        # Convert features into a DataFrame
        column_names = [f"mfcc{i}" for i in range(80)] 
        features_df = pd.DataFrame([features], columns=column_names) 
        
        features_df['sid'] = sid
        features_df['filename'] = name
        
        # Merge into the resulting DataFrame
        result_df = pd.concat([result_df, features_df], ignore_index=True)

In [ ]:
result_df

In [ ]:
result_df['sid'].nunique()

In [ ]:
result_df.to_pickle('******mfcc_features_fullset9071.pkl')

### eGemaps extraction

In [ ]:
# read full metadata from xlxs file
import pandas as pd
import sys

meta_full = pd.read_excel('*********_clinicalinterpretation.xlsx')
meta_full

In [ ]:
meta_full['metadata'].value_counts()

In [ ]:
meta_full_speech = meta_full[
    meta_full['metadata'].str.contains('read_text|answer_question', na=False)
]

meta_full_speech

In [ ]:
import os
import pandas as pd
import opensmile
import librosa


# Initialize the eGeMAPSv01b extractor
smile = opensmile.Smile(
            feature_set=opensmile.FeatureSet.eGeMAPSv01b,
            feature_level=opensmile.FeatureLevel.Functionals,
        )

root_dir = "********/tacticas_audio/tacticas" 


result_df = pd.DataFrame()

#iterate though ROWS IN meta_id_speech
for index, row in meta_full_speech.iterrows():
    name = row['filename'].split('/')[-1]
    sid = row['sid']
    path = f'uid0000{sid}/{name}' 
    audio_path = os.path.join(root_dir, path)
    if os.path.exists(audio_path):
        # Read audio files
        audio, sr = librosa.load(audio_path)
        
        # Extract features
        features = smile.process_signal(audio, sr)
        
        # Convert features into a DataFrame
        features_df = pd.DataFrame(features)
        
        features_df['sid'] = sid
        features_df['filename'] = name
        
        # Merge into the resulting DataFrame
        result_df = pd.concat([result_df, features_df], ignore_index=True)

In [ ]:
result_df

In [ ]:
result_df.to_pickle('*********egemaps_features_speech9071.pkl')

### Concatenate features

In [ ]:
# read pkl file
import pandas as pd
df_mfcc = pd.read_pickle('******mfcc_features_fullset9071.pkl')
df_mfcc

In [ ]:
df_egemaps = pd.read_pickle('********egemaps_features_speech9071.pkl')
df_egemaps

In [ ]:

merged_df = pd.merge(df_egemaps, df_mfcc, on=['filename','sid'], how='outer')
merged_df

In [ ]:
merged_df = merged_df[['sid'] + [col for col in merged_df.columns if col != 'sid']]
merged_df = merged_df[['filename'] + [col for col in merged_df.columns if col != 'filename']]

In [ ]:
merged_df

In [ ]:
merged_df['filename_sid'] = merged_df['sid'].astype(str)  + '_' + merged_df['filename'] 
merged_df = merged_df[['filename_sid'] + [col for col in merged_df.columns if col != 'filename_sid']]
merged_df

In [ ]:
meta_full = pd.read_excel('***********_clinicalinterpretation.xlsx')
meta_full

In [ ]:
meta_full['filename_sid'] = (meta_full['sid'].astype(str) + '_' + meta_full['filename'].str.split('/').str[-1])
meta_full

In [ ]:
meta_full_select = meta_full[['filename_sid', 'event_clinical_yesno']]
meta_full_select

In [ ]:
# merge merged_df and meta_full_select
final_df = pd.merge(merged_df, meta_full_select, on='filename_sid', how='inner')
final_df

In [ ]:
final_df['event_clinical_yesno'].value_counts()

In [ ]:
final_df = final_df[['event_clinical_yesno'] + [col for col in final_df.columns if col != 'event_clinical_yesno']]

In [ ]:
final_df

In [ ]:
final_df_select1  = final_df[['shimmerLocaldB_sma3nz_amean', 'shimmerLocaldB_sma3nz_stddevNorm',
                                             'loudness_sma3_amean', 'loudness_sma3_stddevNorm', 
                                             'HNRdBACF_sma3nz_amean', 'HNRdBACF_sma3nz_stddevNorm',

                                             'loudnessPeaksPerSec',
                                             'MeanVoicedSegmentLengthSec', 'StddevVoicedSegmentLengthSec',
                                              'MeanUnvoicedSegmentLength', 'StddevUnvoicedSegmentLength',
                                               'VoicedSegmentsPerSec',

                                               'F0semitoneFrom27.5Hz_sma3nz_amean','F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
                                             'jitterLocal_sma3nz_amean', 'jitterLocal_sma3nz_stddevNorm', 
                                              'F1frequency_sma3nz_amean', 'F1frequency_sma3nz_stddevNorm',  
                                              'F2frequency_sma3nz_amean',  'F2frequency_sma3nz_stddevNorm', 
                                               'F3frequency_sma3nz_stddevNorm','F3amplitudeLogRelF0_sma3nz_amean',
                                               'F1bandwidth_sma3nz_amean', 'F1bandwidth_sma3nz_stddevNorm',
                                               
                                               'mfcc0', 'mfcc1', 'mfcc2', 'mfcc3', 'mfcc4', 'mfcc5', 'mfcc6', 'mfcc7',
                                               'mfcc8', 'mfcc9', 'mfcc10', 'mfcc11', 'mfcc12', 'mfcc13', 'mfcc14', 'mfcc15',
                                               'mfcc16', 'mfcc17', 'mfcc18', 'mfcc19', 'mfcc20',
                                               'mfcc21', 'mfcc22', 'mfcc23', 'mfcc24', 'mfcc25', 'mfcc26', 'mfcc27', 'mfcc28',
                                               'mfcc29', 


                                             'sid','filename', 'filename_sid', 'event_clinical_yesno'
                                             
                                             ]]

In [ ]:
final_df_select1

In [ ]:
final_df_select1.columns

In [ ]:
final_df_select1

In [ ]:
meta_df = pd.read_excel('**********Tacticas_info_30-10-2024 copy.xlsx')
meta_df

In [ ]:
# give the 'Illness' column to final_df_select1 based on 'sid'(in meta_df:'zanaid', in final_df_select1:'sid')
final_df_select1 = pd.merge(final_df_select1, meta_df[['zanaid', 'Illness']], left_on='sid', right_on='zanaid', how='left')
final_df_select1 = final_df_select1.drop(columns=['zanaid'])
final_df_select1

In [ ]:
final_df_select1_nona = final_df_select1.dropna(subset=['Illness'])
final_df_select1_nona

In [ ]:
final_df_select1_nona['Illness'].value_counts()

In [ ]:
# create a new column 'label' based on 'Illness', if 'Illness' is 'HC', then 'label' is 0, else 'label' is 1
final_df_select1_nona['label'] = final_df_select1_nona['Illness'].apply(lambda x: 0 if x == 'Astma' else 1)
final_df_select1_nona['label'].value_counts()

In [ ]:
final_df_select1_nona.to_pickle('*********final_df_merge8918.pkl')